# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Procesamiento

In [5]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputamos nulos con la mediana
# (los árboles no necesitan escalado)
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns)
])

# Preprocesador para modelos sensibles a la escala (LogReg y KNN):
# imputamos y luego estandarizamos (media 0, desviación 1)
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si ajustamos el `StandardScaler` con todo el conjunto de entrenamiento antes de la validación cruzada, la media y la desviación estándar se calculan usando también los datos del fold de validación. Eso es **fuga de información**: el modelo "ve" algo de los datos con los que luego se evalúa y la métrica de CV sale optimista. Si el scaler está dentro del `Pipeline`, en cada fold se ajusta **solo con los datos de entrenamiento de ese fold** y después se aplica al fold de validación, igual que pasaría con datos nuevos en producción.

## 3. Modelos candidatos

In [6]:
# Un pipeline por modelo: preprocesamiento + clasificador
models = {
    "decision_tree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", KNeighborsClassifier()),
    ]),
}

models

{'decision_tree': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   SimpleImputer(strategy='median'),
                                                   ['alcohol', 'malic_acid',
                                                    'ash', 'alcalinity_of_ash',
                                                    'magnesium', 'total_phenols',
                                                    'flavanoids',
                                                    'nonflavanoid_phenols',
                                                    'proanthocyanins',
                                                    'color_intensity', 'hue',
                                                    'od280/od315_of_diluted_wines',
                                                    'proline'])])),
                 ('model', DecisionTreeClassifier(random_state=42))]),
 'logistic_regression': Pipeline(steps=[('preprocesso

## 4. Comparación inicial con validación cruzada

In [7]:
# Validación cruzada estratificada de 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy_mean", ascending=False)
initial_comparison

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
1,logistic_regression,1.0000,0.979064,0.027713
2,knn,0.9771,0.950739,0.035274
0,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [8]:
# Espacios de búsqueda pequeños (los nombres usan el prefijo "model__")
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        clone(pipeline), param_grids[name],
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    # Fila del mejor modelo dentro de cv_results_
    best = search.best_index_
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "train_accuracy_mean": search.cv_results_["mean_train_score"][best],
        "val_accuracy_mean": search.best_score_,
        "val_accuracy_std": search.cv_results_["std_test_score"][best],
    })

tuned_comparison = pd.DataFrame(rows).sort_values("val_accuracy_mean", ascending=False)
tuned_comparison

,model,best_params,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
1,logistic_regression,{'model__C': 1},1.000000,0.979064,0.027713
2,knn,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.964773,0.957882,0.040520
0,decision_tree,"{'model__max_depth': 4, 'model__min_samples_le...",0.998246,0.902217,0.058617


## 6. Selección y persistencia del campeón

In [8]:
# El campeón es el modelo con mayor accuracy promedio en validación cruzada
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_   # ya reentrenado con todo train (refit=True)

print("Campeón:", champion_name)
print("Mejores hiperparámetros:", searches[champion_name].best_params_)
print("Accuracy CV:", round(searches[champion_name].best_score_, 4))

# 1) Guardamos el pipeline completo (preprocesamiento + modelo)
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# 2) Metadatos del entrenamiento
best_idx = searches[champion_name].best_index_
training_metadata = {
    "champion": champion_name,
    "best_params": {k: (v if v is None or isinstance(v, (int, float, str)) else str(v))
                    for k, v in searches[champion_name].best_params_.items()},
    "cv_accuracy_mean": float(searches[champion_name].best_score_),
    "cv_accuracy_std": float(searches[champion_name].cv_results_["std_test_score"][best_idx]),
    "cv_folds": 5,
    "features": feature_columns,
    "random_state": RANDOM_STATE,
}
with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2)

# 3) Tabla con la comparación de CV
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

training_metadata

Campeón: logistic_regression
Mejores hiperparámetros: {'model__C': 1}
Accuracy CV: 0.9791


{'champion': 'logistic_regression',
 'best_params': {'model__C': 1},
 'cv_accuracy_mean': 0.9790640394088669,
 'cv_accuracy_std': 0.027713464392187602,
 'cv_folds': 5,
 'features': ['alcohol',
  'malic_acid',
  'ash',
  'alcalinity_of_ash',
  'magnesium',
  'total_phenols',
  'flavanoids',
  'nonflavanoid_phenols',
  'proanthocyanins',
  'color_intensity',
  'hue',
  'od280/od315_of_diluted_wines',
  'proline'],
 'random_state': 42}

In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> **Campeón: Regresión Logística (`C=1`) con imputación + `StandardScaler`.**
>
> - **Desempeño:** fue el mejor en validación cruzada, con accuracy promedio ≈ **0.979**, por encima de KNN (≈ 0.958 con `n_neighbors=7`) y del árbol de decisión (≈ 0.902 con `max_depth=4`).
> - **Variabilidad:** también fue el más estable entre folds: desviación estándar ≈ 0.028, contra ≈ 0.041 de KNN y ≈ 0.059 del árbol.
> - **Sobreajuste:** su accuracy en train es 1.0, pero la de validación casi no baja (0.98), así que la brecha es pequeña. El árbol sí muestra más sobreajuste: casi 1.0 en train y solo 0.90 en validación.
> - **Complejidad:** es un modelo lineal y sencillo, rápido de entrenar y con un solo hiperparámetro importante (`C`).
> - **Interpretabilidad:** sus coeficientes (sobre variables estandarizadas) indican qué variables empujan hacia cada clase. Es menos visual que un árbol, pero más interpretable que KNN.
>
> Por todo esto es la opción con mejor balance entre rendimiento, estabilidad y simplicidad.
